# Create 1:1 Matched Control Dataset for C4

This notebook creates a 1:1 matched case-control dataset from C4, matching:
- Age (within ±2 years)
- Sex (exact match)
- Other demographic variables (if available)

**Method**: Nearest neighbor matching with caliper on age

In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
import os

# Paths
C4_PROCESSED_PATH = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_processed.csv'
C4_BALANCED_PATH = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv'
OUTPUT_PATH = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_1to1_matched.csv'

np.random.seed(42)

print("✅ Paths configured")

## Step 1: Load C4 Dataset

In [ ]:
print("="*80)
print("LOADING C4 DATASET")
print("="*80)

# Load processed C4 data
df = pd.read_csv(C4_PROCESSED_PATH)

print(f"\nDataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

# Check autism target distribution
if 'autism_target' in df.columns:
    print(f"\nAutism target distribution:")
    print(df['autism_target'].value_counts())
    print(f"Autism prevalence: {df['autism_target'].mean()*100:.2f}%")
else:
    print("\n⚠️  No 'autism_target' column found")
    print("   Available columns:", [c for c in df.columns if 'autism' in c.lower() or 'diagnos' in c.lower()])

print(f"\n✅ Dataset loaded")

## Step 2: Prepare Matching Variables

In [ ]:
print("="*80)
print("PREPARING MATCHING VARIABLES")
print("="*80)

# Ensure we have autism_target
if 'autism_target' not in df.columns:
    # Try to create it from diagnosis columns
    diagnosis_cols = [c for c in df.columns if 'diagnos' in c.lower() or 'autism' in c.lower()]
    if diagnosis_cols:
        diagnosis_col = diagnosis_cols[0]
        df['autism_target'] = (df[diagnosis_col] == 1).astype(int)
        print(f"Created autism_target from '{diagnosis_col}'")
    else:
        raise ValueError("Cannot find autism_target or diagnosis column")

# Separate cases and controls
cases = df[df['autism_target'] == 1].copy()
controls = df[df['autism_target'] == 0].copy()

print(f"\nCases (autism): {len(cases)}")
print(f"Controls (non-autism): {len(controls)}")

# Prepare matching variables
# Age (required)
if 'age' not in df.columns:
    # Try to find age column
    age_cols = [c for c in df.columns if 'age' in c.lower()]
    if age_cols:
        df['age'] = pd.to_numeric(df[age_cols[0]], errors='coerce')
        print(f"Created 'age' from '{age_cols[0]}'")
    else:
        raise ValueError("Cannot find age column")

# Sex (required)
if 'sex' not in df.columns:
    # Try to find sex column
    sex_cols = [c for c in df.columns if 'sex' in c.lower() or 'gender' in c.lower()]
    if sex_cols:
        df['sex'] = df[sex_cols[0]]
        print(f"Created 'sex' from '{sex_cols[0]}'")
    else:
        print("⚠️  No sex column found - will match only on age")
        df['sex'] = 0  # Dummy value

# Handle missing values
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['age'] = df['age'].fillna(df['age'].median())

# Ensure sex is numeric
if df['sex'].dtype == 'object':
    sex_mapping = {'male': 1, 'female': 2, 'm': 1, 'f': 2, '1': 1, '2': 2}
    df['sex'] = df['sex'].astype(str).str.lower().map(sex_mapping).fillna(0)

df['sex'] = pd.to_numeric(df['sex'], errors='coerce').fillna(0)

# Update cases and controls
cases = df[df['autism_target'] == 1].copy()
controls = df[df['autism_target'] == 0].copy()

print(f"\nAge statistics:")
print(f"  Cases: mean={cases['age'].mean():.1f}, std={cases['age'].std():.1f}, range={cases['age'].min():.0f}-{cases['age'].max():.0f}")
print(f"  Controls: mean={controls['age'].mean():.1f}, std={controls['age'].std():.1f}, range={controls['age'].min():.0f}-{controls['age'].max():.0f}")

print(f"\nSex distribution:")
print(f"  Cases: {cases['sex'].value_counts().to_dict()}")
print(f"  Controls: {controls['sex'].value_counts().to_dict()}")

print(f"\n✅ Matching variables prepared")

## Step 3: Perform 1:1 Matching

In [ ]:
print("="*80)
print("PERFORMING 1:1 MATCHING")
print("="*80)

# Matching parameters
AGE_CALIPER = 2.0  # Match within ±2 years
MATCH_ON_SEX = True  # Require exact sex match

# Prepare matching features
matching_features = ['age']
if MATCH_ON_SEX:
    matching_features.append('sex')

# Prepare case and control feature matrices
X_cases = cases[matching_features].values
X_controls = controls[matching_features].values

# Standardize features for distance calculation
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_cases_scaled = scaler.fit_transform(X_cases)
X_controls_scaled = scaler.transform(X_controls)

# Find nearest neighbors
print(f"\nFinding nearest neighbors for {len(cases)} cases...")
nn = NearestNeighbors(n_neighbors=min(10, len(controls)), metric='euclidean')
nn.fit(X_controls_scaled)

# Match each case to nearest control
matched_pairs = []
matched_control_indices = set()

for i, case in enumerate(cases.itertuples()):
    case_features = X_cases_scaled[i:i+1]
    
    # Find nearest neighbors
    distances, indices = nn.kneighbors(case_features, n_neighbors=min(10, len(controls)))
    
    # Find best match that:
    # 1. Hasn't been matched yet
    # 2. Matches sex (if required)
    # 3. Is within age caliper
    best_match_idx = None
    
    for j, control_idx in enumerate(indices[0]):
        control_row = controls.iloc[control_idx]
        
        # Check if already matched
        if control_idx in matched_control_indices:
            continue
        
        # Check sex match
        if MATCH_ON_SEX and case.sex != control_row['sex']:
            continue
        
        # Check age caliper
        age_diff = abs(case.age - control_row['age'])
        if age_diff > AGE_CALIPER:
            continue
        
        # This is a valid match
        best_match_idx = control_idx
        break
    
    if best_match_idx is not None:
        matched_pairs.append((case.Index, controls.iloc[best_match_idx].name))
        matched_control_indices.add(best_match_idx)
    
    if (i + 1) % 1000 == 0:
        print(f"  Matched {i + 1}/{len(cases)} cases...")

print(f"\n✅ Matching complete")
print(f"   Matched pairs: {len(matched_pairs)}")
print(f"   Match rate: {len(matched_pairs)/len(cases)*100:.1f}%")

## Step 4: Create Matched Dataset

In [ ]:
print("="*80)
print("CREATING MATCHED DATASET")
print("="*80)

# Extract matched cases and controls
matched_case_indices = [pair[0] for pair in matched_pairs]
matched_control_indices_list = [pair[1] for pair in matched_pairs]

matched_cases = df.loc[matched_case_indices].copy()
matched_controls = df.loc[matched_control_indices_list].copy()

# Combine
df_matched = pd.concat([matched_cases, matched_controls], ignore_index=True)

# Shuffle
df_matched = df_matched.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nMatched dataset shape: {df_matched.shape}")
print(f"\nAutism target distribution:")
print(df_matched['autism_target'].value_counts())
print(f"Balance ratio: {(df_matched['autism_target'] == 0).sum()}:{(df_matched['autism_target'] == 1).sum()}")

# Verify matching quality
print(f"\nMatching quality:")
matched_cases_age = matched_cases['age'].values
matched_controls_age = matched_controls['age'].values
age_diffs = np.abs(matched_cases_age - matched_controls_age)

print(f"  Age differences:")
print(f"    Mean: {age_diffs.mean():.2f} years")
print(f"    Median: {np.median(age_diffs):.2f} years")
print(f"    Max: {age_diffs.max():.2f} years")
print(f"    Within caliper (±{AGE_CALIPER}): {(age_diffs <= AGE_CALIPER).sum()}/{len(age_diffs)} ({(age_diffs <= AGE_CALIPER).mean()*100:.1f}%)")

if MATCH_ON_SEX:
    sex_matches = (matched_cases['sex'].values == matched_controls['sex'].values).sum()
    print(f"  Sex matches: {sex_matches}/{len(matched_pairs)} ({sex_matches/len(matched_pairs)*100:.1f}%)")

print(f"\n✅ Matched dataset created")

## Step 5: Save Matched Dataset

In [ ]:
print("="*80)
print("SAVING MATCHED DATASET")
print("="*80)

# Save
df_matched.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Matched dataset saved to: {OUTPUT_PATH}")
print(f"   Shape: {df_matched.shape}")
print(f"   Balance: {(df_matched['autism_target'] == 0).sum()} controls : {(df_matched['autism_target'] == 1).sum()} cases")

# Summary statistics
print(f"\nSummary:")
print(f"  Original dataset: {len(df)} participants")
print(f"  Original cases: {len(cases)}")
print(f"  Original controls: {len(controls)}")
print(f"  Matched cases: {len(matched_cases)}")
print(f"  Matched controls: {len(matched_controls)}")
print(f"  Final dataset: {len(df_matched)} participants (50/50 split)")

print(f"\n✅ Complete!")